
# Subject transition matrices: decade-level outflow and inflow

This notebook studies advisor–student subject transitions using **full decades**.

For decade \(t\), let

\[
M_t(i,j)
=
\text{weighted number of advisor–student edges with advisor subject }i
\text{ and student subject }j.
\]

Two conditional probability matrices are used.

### Forward matrix \(P_t\): outflow from advisor subjects

\[
P_t(i,j)
=
P(\text{student subject}=j\mid \text{advisor subject}=i,\ t).
\]

Rows are advisor subjects and columns are student subjects. Therefore:

- \(P_t(i,i)\): forward self-retention;
- \(1-P_t(i,i)\): outflow rate from subject \(i\) to other subjects;
- row \(i\): destination distribution of students advised from subject \(i\).

### Backward matrix \(Q_t\): inflow into student subjects

\[
Q_t(j,i)
=
P(\text{advisor subject}=i\mid \text{student subject}=j,\ t).
\]

Rows are student subjects and columns are advisor subjects. Therefore:

- \(Q_t(j,j)\): same-subject ancestry rate;
- \(1-Q_t(j,j)\): inflow rate into subject \(j\) from other advisor subjects;
- row \(j\): source distribution of advisors for students in subject \(j\).

The backward matrix is defined as

\[
Q_t = \operatorname{row\_normalize}(M_t^\top).
\]

This is equivalent to column-normalizing \(M_t\) and then transposing it.

### Why decades rather than individual years?

Yearly matrices answer the same questions, but many subject rows have few or no observations in a single year. This produces unstable probabilities, many undefined rows, and visually noisy trends. Decade aggregation sacrifices some time resolution but gives more reliable transition estimates.

This notebook does **not** multiply matrices across decades. Each matrix is treated as a cohort-specific empirical conditional distribution, not as an automatically composable historical dynamical operator.


In [ ]:

from pathlib import Path
import ast
import json
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)



## 1. Configuration

The primary analysis uses the original `subject` labels. Genealogy-based imputed subject labels should be reserved for a separately labeled sensitivity analysis because using advisor/descendant information in imputation can mechanically strengthen apparent subject transmission.


In [ ]:

# From notebooks/subject_analysis-Yilong/, the repository data path is usually:
DATA_PATH = Path("../../data/processed/data-new.json")

# Analysis window: full decades from 1900–1909 through 2010–2019.
START_YEAR = 1900
END_YEAR_EXCLUSIVE = 2020

# Keep False to reproduce the original edge-count interpretation:
# each observed advisor–student relationship contributes weight 1.
#
# Set True if each student should contribute total weight 1 across all
# of that student's valid advisors.
WEIGHT_MULTIPLE_ADVISORS = False

# Original subject label column.
SUBJECT_COLUMN = "subject"

# Selected subjects for summary plots.
SELECTED_SUBJECTS = ["11", "14", "32", "35", "53", "55", "60", "62", "68"]
TARGET_SUBJECT = "14"


In [ ]:

# Search a few common repository-relative locations if DATA_PATH is not found.
candidates = [
    DATA_PATH,
    Path("../data/processed/data-new.json"),
    Path("data/processed/data-new.json"),
    Path("../../data/processed/version2_new_dataset_fitted.csv"),
]

existing = [path for path in candidates if path.exists()]

if not existing:
    raise FileNotFoundError(
        "Could not find the processed dataset. Set DATA_PATH in the configuration "
        "cell to data-new.json or version2_new_dataset_fitted.csv."
    )

DATA_PATH = existing[0]
print("Loading:", DATA_PATH.resolve())

if DATA_PATH.suffix.lower() == ".json":
    with DATA_PATH.open("r") as file:
        tree = json.load(file)
    nodes = pd.DataFrame(tree["nodes"])
elif DATA_PATH.suffix.lower() == ".csv":
    nodes = pd.read_csv(DATA_PATH, low_memory=False)
else:
    raise ValueError(f"Unsupported file type: {DATA_PATH.suffix}")

print(f"Rows: {len(nodes):,}")
print("Columns:", list(nodes.columns))



## 2. Clean subjects and construct advisor–student edges


In [ ]:

# Standard 2020 MSC top-level codes: 63 subjects
MSC_CODES = [
    "00", "01", "03", "05", "06", "08",
    "11", "12", "13", "14", "15", "16", "17", "18", "19",
    "20", "22", "26", "28", "30", "31", "32", "33", "34", "35", "37", "39",
    "40", "41", "42", "43", "44", "45", "46", "47", "49",
    "51", "52", "53", "54", "55", "57", "58",
    "60", "62", "65", "68",
    "70", "74", "76", "78", "80", "81", "82", "83", "85", "86",
    "90", "91", "92", "93", "94", "97",
]

MSC_LABELS = {
    "11": "11 Number theory",
    "14": "14 Algebraic geometry",
    "18": "18 Category theory",
    "32": "32 Several complex variables",
    "35": "35 Partial differential equations",
    "53": "53 Differential geometry",
    "55": "55 Algebraic topology",
    "57": "57 Manifolds and cell complexes",
    "58": "58 Global analysis",
    "60": "60 Probability theory",
    "62": "62 Statistics",
    "65": "65 Numerical analysis",
    "68": "68 Computer science",
    "81": "81 Quantum theory",
    "90": "90 Operations research",
    "91": "91 Game theory and economics",
}

def label_subject(code):
    code = str(code)
    return MSC_LABELS.get(code, code)


def parse_id_list(value):
    """Convert an advisors entry into a list of integer IDs."""
    if isinstance(value, (list, tuple, set, np.ndarray)):
        result = []
        for item in value:
            try:
                if pd.notna(item):
                    result.append(int(item))
            except (TypeError, ValueError):
                pass
        return result

    if pd.isna(value):
        return []

    if isinstance(value, str):
        text = value.strip()
        if text in {"", "[]", "nan", "None"}:
            return []

        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, (list, tuple, set, np.ndarray)):
                return parse_id_list(parsed)
            return [int(parsed)]
        except (ValueError, SyntaxError, TypeError):
            return [int(item) for item in re.findall(r"\d+", text)]

    return []


def extract_top_msc_code(value):
    """Extract a two-digit top-level MSC code."""
    if pd.isna(value):
        return np.nan

    text = re.sub(r"\.0$", "", str(value).strip())
    match = re.search(r"\d{1,2}", text)

    if match is None:
        return np.nan

    return match.group(0).zfill(2)


In [ ]:

required_columns = {"id", "year", SUBJECT_COLUMN, "advisors"}
missing_columns = required_columns.difference(nodes.columns)

if missing_columns:
    raise KeyError(f"Missing required columns: {sorted(missing_columns)}")

nodes_clean = nodes.copy()
nodes_clean["id_clean"] = pd.to_numeric(nodes_clean["id"], errors="coerce")
nodes_clean["year_clean"] = pd.to_numeric(nodes_clean["year"], errors="coerce")
nodes_clean["subject_code"] = nodes_clean[SUBJECT_COLUMN].apply(extract_top_msc_code)
nodes_clean["advisor_ids"] = nodes_clean["advisors"].apply(parse_id_list)

id_to_subject = (
    nodes_clean.dropna(subset=["id_clean"])
    .assign(id_clean=lambda frame: frame["id_clean"].astype(int))
    .set_index("id_clean")["subject_code"]
    .to_dict()
)

student_rows = nodes_clean[
    nodes_clean["id_clean"].notna()
    & nodes_clean["year_clean"].notna()
    & nodes_clean["subject_code"].isin(MSC_CODES)
    & nodes_clean["advisor_ids"].map(len).gt(0)
][["id_clean", "year_clean", "subject_code", "advisor_ids"]].copy()

edges = (
    student_rows
    .explode("advisor_ids")
    .rename(
        columns={
            "id_clean": "student_id",
            "year_clean": "student_year",
            "subject_code": "student_subject",
            "advisor_ids": "advisor_id",
        }
    )
)

edges["student_id"] = edges["student_id"].astype(int)
edges["advisor_id"] = pd.to_numeric(edges["advisor_id"], errors="coerce")
edges = edges.dropna(subset=["advisor_id"]).copy()
edges["advisor_id"] = edges["advisor_id"].astype(int)
edges["advisor_subject"] = edges["advisor_id"].map(id_to_subject)

edges = edges[
    edges["advisor_subject"].isin(MSC_CODES)
    & edges["student_subject"].isin(MSC_CODES)
].copy()

edges["student_year"] = edges["student_year"].astype(int)
edges = edges[
    edges["student_year"].between(
        START_YEAR,
        END_YEAR_EXCLUSIVE - 1,
        inclusive="both",
    )
].copy()

edges["decade"] = (edges["student_year"] // 10) * 10

if WEIGHT_MULTIPLE_ADVISORS:
    valid_advisor_count = edges.groupby("student_id")["advisor_id"].transform("count")
    edges["weight"] = 1.0 / valid_advisor_count
else:
    edges["weight"] = 1.0

print(f"Valid advisor–student edge rows: {len(edges):,}")
print(f"Unique students represented: {edges['student_id'].nunique():,}")
print(f"Weighted edge total: {edges['weight'].sum():,.1f}")
edges.head()



## 3. Build the decade count matrices \(M_t\)

Rows are advisor subjects and columns are student subjects. The decade is determined by the student's PhD year.


In [ ]:

DECADES = list(range(START_YEAR, END_YEAR_EXCLUSIVE, 10))

def transition_matrix_for_decade(edge_data, decade, codes=MSC_CODES):
    """Return a subject-by-subject weighted transition count matrix."""
    subset = edge_data[edge_data["decade"] == decade]

    matrix = pd.pivot_table(
        subset,
        values="weight",
        index="advisor_subject",
        columns="student_subject",
        aggfunc="sum",
        fill_value=0.0,
    )

    return matrix.reindex(index=codes, columns=codes, fill_value=0.0).astype(float)


M_by_decade = {
    decade: transition_matrix_for_decade(edges, decade)
    for decade in DECADES
}

M_by_decade[1960].iloc[:8, :8]



## 4. Define the forward matrix \(P_t\) and backward matrix \(Q_t\)

Rows with no observations are left as `NaN`, because their conditional probability distribution is undefined. Treating an unsupported row as all zeros and then computing \(1-P_{ii}\) would incorrectly report a 100% outflow rate.


In [ ]:

def normalize_rows(matrix):
    """Normalize nonzero rows to sum to one; leave zero rows undefined."""
    row_totals = matrix.sum(axis=1)
    return matrix.div(row_totals.replace(0, np.nan), axis=0)


def forward_matrix(matrix):
    """
    P[i, j] = Pr(student subject j | advisor subject i).
    Rows: advisor subjects. Columns: student subjects.
    """
    return normalize_rows(matrix)


def backward_matrix(matrix):
    """
    Q[j, i] = Pr(advisor subject i | student subject j).
    Rows: student subjects. Columns: advisor subjects.
    """
    return normalize_rows(matrix.T)


P_by_decade = {
    decade: forward_matrix(matrix)
    for decade, matrix in M_by_decade.items()
}

Q_by_decade = {
    decade: backward_matrix(matrix)
    for decade, matrix in M_by_decade.items()
}

# Verification: every supported row should sum to one.
def supported_row_sums(matrix):
    sums = matrix.sum(axis=1, min_count=1)
    return sums[sums.notna()]

for name, matrices in {"P": P_by_decade, "Q": Q_by_decade}.items():
    for decade, matrix in matrices.items():
        sums = supported_row_sums(matrix)
        if len(sums) and not np.allclose(sums.to_numpy(), 1.0):
            raise AssertionError(f"{name}_{decade} has a supported row not summing to 1.")

print("All supported rows of P and Q sum to 1.")



## 5. Data-support diagnostics

These quantities should accompany probability trends. A decade with few edges or many unsupported subjects will produce less stable estimates.


In [ ]:

def support_summary(matrix):
    row_totals = matrix.sum(axis=1)
    column_totals = matrix.sum(axis=0)

    return {
        "weighted_edge_total": matrix.to_numpy().sum(),
        "active_advisor_subjects": int((row_totals > 0).sum()),
        "active_student_subjects": int((column_totals > 0).sum()),
        "zero_advisor_rows": int((row_totals == 0).sum()),
        "zero_student_columns": int((column_totals == 0).sum()),
        "rows_with_fewer_than_10_edges": int(
            ((row_totals > 0) & (row_totals < 10)).sum()
        ),
        "matrix_density": float((matrix.to_numpy() > 0).mean()),
    }


support_df = pd.DataFrame(
    {
        decade: support_summary(matrix)
        for decade, matrix in M_by_decade.items()
    }
).T

support_df.index.name = "decade"
support_df


In [ ]:

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(
    support_df.index,
    support_df["weighted_edge_total"],
    marker="o",
)
ax.set_xlabel("Decade")
ax.set_ylabel("Weighted advisor–student edge total")
ax.set_title("Data support by decade")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()



## 6. Subject-level retention, outflow, and inflow rates

For subject \(i\):

\[
\text{forward retention}_i(t)=P_t(i,i),
\qquad
\text{outflow rate}_i(t)=1-P_t(i,i),
\]

and

\[
\text{same-subject ancestry}_i(t)=Q_t(i,i),
\qquad
\text{inflow rate}_i(t)=1-Q_t(i,i).
\]

The outflow rate is the proportion of students advised from subject \(i\) who move to another subject.

The inflow rate is the proportion of students in subject \(i\) whose advisors came from another subject.


In [ ]:

def diagonal_frame(matrices):
    frame = pd.DataFrame(
        {
            decade: pd.Series(
                np.diag(matrix.to_numpy()),
                index=matrix.index,
            )
            for decade, matrix in matrices.items()
        }
    ).T
    frame.index.name = "decade"
    frame.columns.name = "subject"
    return frame


forward_retention_df = diagonal_frame(P_by_decade)
same_subject_ancestry_df = diagonal_frame(Q_by_decade)

outflow_rate_df = 1.0 - forward_retention_df
inflow_rate_df = 1.0 - same_subject_ancestry_df

# Unsupported subject-decades remain NaN rather than being reported as 100%.
outflow_rate_df.head()


In [ ]:

def plot_subject_rates(subjects=SELECTED_SUBJECTS):
    fig, axes = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(11, 10),
        sharex=True,
    )

    for subject in subjects:
        if subject in outflow_rate_df.columns:
            axes[0].plot(
                outflow_rate_df.index,
                outflow_rate_df[subject],
                marker="o",
                label=label_subject(subject),
            )

        if subject in inflow_rate_df.columns:
            axes[1].plot(
                inflow_rate_df.index,
                inflow_rate_df[subject],
                marker="o",
                label=label_subject(subject),
            )

    axes[0].set_ylabel("Outflow rate")
    axes[0].set_title(
        "Forward outflow: share of students leaving their advisors' subject"
    )

    axes[1].set_xlabel("Decade")
    axes[1].set_ylabel("Inflow rate")
    axes[1].set_title(
        "Backward inflow: share of students whose advisors came from other subjects"
    )

    for axis in axes:
        axis.set_ylim(0, 1)
        axis.grid(alpha=0.3)
        axis.legend(bbox_to_anchor=(1.02, 1), loc="upper left")

    fig.tight_layout()
    plt.show()


plot_subject_rates()



## 7. Raw flow counts

The probability matrices answer conditional questions. The count matrix answers volume questions.

- Raw outflow count from subject \(i\): off-diagonal sum of row \(i\).
- Raw inflow count into subject \(i\): off-diagonal sum of column \(i\).
- Net inflow: raw inflow minus raw outflow.

These counts should not be called rates.


In [ ]:

def flow_count_summary(matrix):
    diagonal = pd.Series(
        np.diag(matrix.to_numpy()),
        index=matrix.index,
        dtype=float,
    )

    raw_outflow = matrix.sum(axis=1) - diagonal
    raw_inflow = matrix.sum(axis=0) - diagonal

    return pd.DataFrame(
        {
            "raw_inflow_count": raw_inflow,
            "raw_outflow_count": raw_outflow,
            "net_inflow_count": raw_inflow - raw_outflow,
            "same_subject_count": diagonal,
            "advisor_edge_total": matrix.sum(axis=1),
            "student_edge_total": matrix.sum(axis=0),
        }
    )


flow_counts_by_decade = {
    decade: flow_count_summary(matrix)
    for decade, matrix in M_by_decade.items()
}

flow_counts_by_decade[1960].sort_values(
    "net_inflow_count",
    ascending=False,
).head(10)


In [ ]:

def target_count_timeseries(target=TARGET_SUBJECT):
    rows = []

    for decade, summary in flow_counts_by_decade.items():
        row = summary.loc[target]
        rows.append(
            {
                "decade": decade,
                "raw_inflow_count": row["raw_inflow_count"],
                "raw_outflow_count": row["raw_outflow_count"],
                "net_inflow_count": row["net_inflow_count"],
                "same_subject_count": row["same_subject_count"],
            }
        )

    return pd.DataFrame(rows).sort_values("decade")


target_counts = target_count_timeseries()

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(
    target_counts["decade"],
    target_counts["raw_inflow_count"],
    marker="o",
    label="Raw inflow count",
)
ax.plot(
    target_counts["decade"],
    target_counts["raw_outflow_count"],
    marker="o",
    label="Raw outflow count",
)
ax.plot(
    target_counts["decade"],
    target_counts["net_inflow_count"],
    marker="o",
    label="Net inflow count",
)
ax.axhline(0, linewidth=0.8)
ax.set_xlabel("Decade")
ax.set_ylabel("Weighted advisor–student edge count")
ax.set_title(f"Raw flow volume around {label_subject(TARGET_SUBJECT)}")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()



## 8. Destination and source profiles for a selected subject

- Use a row of \(P_t\) for the destinations of students advised from the selected subject.
- Use a row of \(Q_t\) for the advisor sources of students entering the selected subject.


In [ ]:

def top_outflow_destinations(
    decade,
    source=TARGET_SUBJECT,
    n=10,
    exclude_self=True,
):
    probabilities = P_by_decade[decade].loc[source].dropna()

    if exclude_self:
        probabilities = probabilities.drop(source, errors="ignore")

    result = (
        probabilities
        .sort_values(ascending=False)
        .head(n)
        .rename("outflow_probability")
        .reset_index()
    )

    result = result.rename(columns={result.columns[0]: "destination_subject"})
    result["destination_label"] = result["destination_subject"].map(label_subject)
    return result


def top_inflow_sources(
    decade,
    target=TARGET_SUBJECT,
    n=10,
    exclude_self=True,
):
    probabilities = Q_by_decade[decade].loc[target].dropna()

    if exclude_self:
        probabilities = probabilities.drop(target, errors="ignore")

    result = (
        probabilities
        .sort_values(ascending=False)
        .head(n)
        .rename("inflow_probability")
        .reset_index()
    )

    result = result.rename(columns={result.columns[0]: "source_subject"})
    result["source_label"] = result["source_subject"].map(label_subject)
    return result


display(top_outflow_destinations(1960))
display(top_inflow_sources(1960))


In [ ]:

def probability_profile_over_time(
    matrices,
    subject,
    top_n=12,
    exclude_self=True,
):
    """
    Extract one row from each decade matrix.

    For P matrices:
        subject is an advisor subject and columns are student destinations.
    For Q matrices:
        subject is a student subject and columns are advisor sources.
    """
    profile = pd.DataFrame(
        {
            decade: matrix.loc[subject]
            for decade, matrix in matrices.items()
        }
    )

    if exclude_self:
        profile = profile.drop(index=subject, errors="ignore")

    ranking = profile.fillna(0).sum(axis=1).sort_values(ascending=False)
    return profile.loc[ranking.head(top_n).index]


def plot_probability_heatmap(
    profile,
    title,
    colorbar_label,
):
    plot_data = profile.fillna(0)
    labels = [label_subject(code) for code in plot_data.index]

    fig, ax = plt.subplots(figsize=(11, 7))
    image = ax.imshow(plot_data.to_numpy(), aspect="auto")
    fig.colorbar(image, ax=ax, label=colorbar_label)

    ax.set_xticks(np.arange(len(plot_data.columns)))
    ax.set_xticklabels(plot_data.columns, rotation=45)

    ax.set_yticks(np.arange(len(plot_data.index)))
    ax.set_yticklabels(labels)

    ax.set_xlabel("Decade")
    ax.set_title(title)
    fig.tight_layout()
    plt.show()


outflow_profile = probability_profile_over_time(
    P_by_decade,
    TARGET_SUBJECT,
)

inflow_profile = probability_profile_over_time(
    Q_by_decade,
    TARGET_SUBJECT,
)

plot_probability_heatmap(
    outflow_profile,
    title=f"Destinations from {label_subject(TARGET_SUBJECT)} using P",
    colorbar_label="Conditional outflow probability",
)

plot_probability_heatmap(
    inflow_profile,
    title=f"Advisor sources into {label_subject(TARGET_SUBJECT)} using Q",
    colorbar_label="Conditional inflow probability",
)



## 9. Decade-level features for inferential analysis

The following features summarize the whole transition table without relying on matrix composition.

### Observed retention

\[
R_t
=
\frac{\sum_i M_t(i,i)}
{\sum_{i,j}M_t(i,j)}.
\]

### Retention expected under independence

Let \(a_i\) and \(s_i\) be the advisor- and student-subject marginal proportions. Under independence,

\[
R_t^{\mathrm{null}}=\sum_i a_i s_i.
\]

### Excess retention and Cohen's kappa

\[
R_t-R_t^{\mathrm{null}},
\qquad
\kappa_t
=
\frac{R_t-R_t^{\mathrm{null}}}
{1-R_t^{\mathrm{null}}}.
\]

### Mutual information

Mutual information uses the full table, not only its diagonal, and measures how informative advisor subject is about student subject.

### Weighted forward entropy

This measures the average diversity of student destinations, weighted by the number of advisor edges supporting each row.


In [ ]:

def entropy(probabilities):
    probabilities = np.asarray(probabilities, dtype=float)
    probabilities = probabilities[
        np.isfinite(probabilities) & (probabilities > 0)
    ]

    if len(probabilities) == 0:
        return np.nan

    return float(-(probabilities * np.log(probabilities)).sum())


def decade_features(matrix):
    counts = matrix.to_numpy(dtype=float)
    total = counts.sum()

    if total <= 0:
        return {
            "observed_retention": np.nan,
            "expected_retention": np.nan,
            "excess_retention": np.nan,
            "cohens_kappa": np.nan,
            "mutual_information": np.nan,
            "normalized_mutual_information": np.nan,
            "weighted_forward_entropy": np.nan,
        }

    joint = counts / total
    advisor_marginal = joint.sum(axis=1)
    student_marginal = joint.sum(axis=0)

    observed_retention = float(np.trace(joint))
    expected_retention = float(
        np.sum(advisor_marginal * student_marginal)
    )

    denominator = 1.0 - expected_retention
    cohens_kappa = (
        (observed_retention - expected_retention) / denominator
        if denominator > 0
        else np.nan
    )

    independent_joint = np.outer(advisor_marginal, student_marginal)
    mask = (joint > 0) & (independent_joint > 0)

    mutual_information = float(
        np.sum(
            joint[mask]
            * np.log(joint[mask] / independent_joint[mask])
        )
    )

    advisor_entropy = entropy(advisor_marginal)
    student_entropy = entropy(student_marginal)

    if (
        np.isfinite(advisor_entropy)
        and np.isfinite(student_entropy)
        and advisor_entropy > 0
        and student_entropy > 0
    ):
        normalized_mi = mutual_information / np.sqrt(
            advisor_entropy * student_entropy
        )
    else:
        normalized_mi = np.nan

    P = forward_matrix(matrix)
    row_entropies = P.apply(
        lambda row: entropy(row.to_numpy()),
        axis=1,
    )

    advisor_weights = matrix.sum(axis=1) / total
    valid = row_entropies.notna() & (advisor_weights > 0)

    weighted_forward_entropy = float(
        np.sum(
            advisor_weights[valid]
            * row_entropies[valid]
        )
    )

    return {
        "observed_retention": observed_retention,
        "expected_retention": expected_retention,
        "excess_retention": observed_retention - expected_retention,
        "cohens_kappa": cohens_kappa,
        "mutual_information": mutual_information,
        "normalized_mutual_information": normalized_mi,
        "weighted_forward_entropy": weighted_forward_entropy,
    }


feature_df = pd.DataFrame(
    {
        decade: decade_features(matrix)
        for decade, matrix in M_by_decade.items()
    }
).T

feature_df.index.name = "decade"
feature_df


In [ ]:

fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(10, 12),
    sharex=True,
)

axes[0].plot(
    feature_df.index,
    feature_df["observed_retention"],
    marker="o",
    label="Observed",
)
axes[0].plot(
    feature_df.index,
    feature_df["expected_retention"],
    marker="o",
    label="Expected under independence",
)
axes[0].set_ylabel("Retention rate")
axes[0].set_title("Observed and chance-expected subject retention")
axes[0].legend()

axes[1].plot(
    feature_df.index,
    feature_df["cohens_kappa"],
    marker="o",
)
axes[1].axhline(0, linewidth=0.8)
axes[1].set_ylabel("Cohen's kappa")
axes[1].set_title("Excess subject transmission beyond marginal frequencies")

axes[2].plot(
    feature_df.index,
    feature_df["normalized_mutual_information"],
    marker="o",
    label="Normalized mutual information",
)
axes[2].plot(
    feature_df.index,
    feature_df["weighted_forward_entropy"],
    marker="o",
    label="Weighted forward entropy",
)
axes[2].set_xlabel("Decade")
axes[2].set_ylabel("Information / entropy")
axes[2].set_title("Whole-matrix structure")
axes[2].legend()

for axis in axes:
    axis.grid(alpha=0.3)

fig.tight_layout()
plt.show()



## 10. Long-form tables for later modeling

These tables are convenient inputs for confidence intervals, permutation tests, bootstrap analyses, or regression models.


In [ ]:

def rates_to_long():
    rows = []

    for decade in DECADES:
        for subject in MSC_CODES:
            rows.append(
                {
                    "decade": decade,
                    "subject": subject,
                    "subject_label": label_subject(subject),
                    "forward_retention": forward_retention_df.loc[decade, subject],
                    "outflow_rate": outflow_rate_df.loc[decade, subject],
                    "same_subject_ancestry": same_subject_ancestry_df.loc[decade, subject],
                    "inflow_rate": inflow_rate_df.loc[decade, subject],
                    "advisor_edge_total": M_by_decade[decade].loc[subject].sum(),
                    "student_edge_total": M_by_decade[decade][subject].sum(),
                }
            )

    return pd.DataFrame(rows)


subject_rate_long = rates_to_long()
subject_rate_long.head()


In [ ]:

# Optional exports
#
# OUTPUT_DIR = Path("outputs")
# OUTPUT_DIR.mkdir(exist_ok=True)
#
# support_df.to_csv(OUTPUT_DIR / "transition_support_by_decade.csv")
# feature_df.to_csv(OUTPUT_DIR / "transition_features_by_decade.csv")
# subject_rate_long.to_csv(
#     OUTPUT_DIR / "subject_outflow_inflow_rates_by_decade.csv",
#     index=False,
# )



## Interpretation summary

- Use \(P_t\) for **forward outflow** questions:
  \[
  P_t(i,j)=P(S=j\mid A=i).
  \]

- Use \(Q_t\) for **backward inflow-source** questions:
  \[
  Q_t(j,i)=P(A=i\mid S=j).
  \]

- Use \(M_t\) for raw inflow/outflow **counts**.

- Do not interpret unsupported rows as 100% mutation; their rates are undefined.

- Decade aggregation is retained because yearly matrices have much smaller sample sizes and consequently noisier conditional probabilities.

- The primary inferential targets are one-step quantities such as retention, excess retention, kappa, mutual information, entropy, and changes in subject-specific outflow/inflow rates. Matrix powers and eigenvalues require an additional repeated-transition model and are therefore omitted from the main analysis.
